In [31]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [32]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from controlkit.se3 import SE3

import controlkit.rerun_viz as rrv
from controlkit.kinematics.types import Support, Foothold, Posture
from controlkit.kinematics.robot import Robot, radial_mounts
from controlkit.kinematics.leg4dof import Leg4DOF
from controlkit.kinematics.collision import overlap
from controlkit.utils import sample_mesh
import controlkit.forces as forces

key = jax.random.PRNGKey(0)

In [33]:
def sample_footholds(key, vertices, faces, M=50_000):
    """Sample footholds from a mesh.

    Args:
        key: PRNGKey
        vertices: (V, 3) mesh vertices
        faces: (F, 3) mesh faces
        M: number of samples
    Returns:
        xs: (M, 3) sampled foothold positions
        ns: (M, 3) sampled foothold normals
    """
    key, subkey = jax.random.split(key)
    xs, ns = sample_mesh(subkey, vertices, faces, M=M)
    return Foothold(xs, ns)


obb_collider = jax.vmap(jax.vmap(overlap, (None,0)), (0,None))

def _check_leg_collision(posture, robot):
    body_obb = robot.body_box(posture)
    leg_obb = robot.leg_boxes(posture)

    leg_body_collisions = obb_collider(leg_obb[3,1:].reshape(-1), body_obb[None])

    return leg_body_collisions.any()
 
@jax.jit
def check_leg_collisions(postures, robot):
    return jax.vmap(_check_leg_collision, (0, None))(postures, robot)


def _compute_contact_angle(posture, support):
    dots = jnp.sum(
            robot.contact_vectors(posture)[support.ids] * support.sites.normal, 
        axis=-1)
    angles = jnp.arccos(jnp.clip(dots, -1.0, 1.0))
    return angles


compute_contact_angles = jax.jit(jax.vmap(_compute_contact_angle))

In [34]:
#
# Assemble the robot
#
leg = Leg4DOF.from_lengths(
    jnp.array([0.05, 0.05, 0.2, 0.2]),
    jnp.deg2rad(jnp.array([
        [-90, 90],
        [-180, 180],
        [-150, 150],
        [-150, 150],
    ]))
)

robot = Robot(
    mounts=radial_mounts(4, radius=0.20),
    leg=leg,
)

mjx_model, foot_ids = forces.model_from_robot(robot)
#
# Test Surface mesh and foothold sampling
#
vertices = jnp.array([
    [-1.0, -1.0, 0.0],
    [1.0, -1.0, 0.0],
    [1.0, 1.0, 0.0],
    [-1.0, 1.0, 0.0],
    [1.0,-1.0,2.0],
    [1.0,1.0,2.0],
 ])
faces = jnp.array([
    [0, 1, 2],
    [0, 2, 3],
    [1, 4, 5],
    [1, 5, 2],
])
mesh = (vertices, faces)

key, _ = jax.random.split(key)
footholds = sample_footholds(key, *mesh, M=10_000)


#
# Initial posture
#
body = SE3.from_te(
    jnp.array([.9, 0.0, 1.]),
    jnp.deg2rad(jnp.array([0.0, -90.0, 0.0])),
)
ids = jnp.array([0,1,2])

key, _ = jax.random.split(key)
mask = robot.reach_mask(key, body, footholds)
mask_any = mask.any(axis=-1)
probs = mask.astype(jnp.float32)/mask.sum(axis=0, keepdims=True)

key, subkey = jax.random.split(key)
keys = jax.random.split(subkey, len(ids))
inds = jax.vmap(
        lambda key, i: jax.random.choice(key, footholds.shape[0], p=probs[:,i]))(
            keys, ids)

support = Support(sites=footholds[inds], ids=ids)
ok, posture = robot.sample_posture(key, body, support)


# =========
rrv.init_viewer(connect=8812) 
rrv.set_time(0, timeline="t")
# rrv.log_boxes("boxes", boxes, colors=[(0, 0, 0)])
rrv.log_mesh("mesh", vertices, faces, color=(200, 200, 200))
rrv.log_contacts("footholds_reachable", 
                 footholds.position[mask_any], 
                 footholds.normal[mask_any], size=0.05,
                 color=[(255, 0, 255)], 
                 normal_color=[(0, 0, 255)], show_normals=False)
rrv.log_posture("posture", robot, posture,          
                    foot_radius=0.025, disk_radius=0.025, 
                    disk_thickness=0.03, link_width=0.025, 
                    joint_values=posture.thetas, 
                    joint_cmap="bwr", 
                    joint_min=-np.pi, joint_max=np.pi,
                    body_color=(200, 200, 200), 
                    link_color=(200, 200, 200), 
                    )

In [35]:
shoulders = robot.shoulders(body)
footholds[None].transform(shoulders[:,None]).shape

(4, 10000)

In [36]:
compute_mask = jax.jit(robot.reach_mask)

In [44]:
print(footholds.shape)

K = 2_000

key, _ = jax.random.split(key)
mask = compute_mask(key, body, footholds[:K])
key, _ = jax.random.split(key)
%timeit -r 1 -n 1 jax.block_until_ready(robot.reach_mask(key, body, footholds[:K]))
key, _ = jax.random.split(key)
%timeit -r 1 -n 1 jax.block_until_ready(compute_mask(key, body, footholds[:K]))
mask_any = mask.any(axis=-1)


(10000,)
261 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
98.5 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [45]:
# mask = robot.reach_mask(key, body, footholds)
xyz_limits = jnp.array([
    [-0., 0.],
    [-0., 0.],
    [-0., 0.]
])
rpy_limits = jnp.array([
    [0., 0.],
    [0., 0.],
    [0., 0.]
])


# Note: Depends on footholds
def _sample_posture(key, body, mask, ids):
    
    probs = mask.astype(jnp.float32)/mask.sum(axis=0, keepdims=True)

    key, subkey = jax.random.split(key)
    keys = jax.random.split(subkey, len(ids))
    inds = jax.vmap(
            lambda key, i: jax.random.choice(key, footholds.shape[0], p=probs[:,i]))(
                keys, ids)

    support = Support(sites=footholds[inds], ids=ids)
    ok, posture = robot.sample_posture(key, body, support, 
                        xyz_limits=xyz_limits, rpy_limits=rpy_limits, seq="xyz")

    return ok, posture, support

sample_posture = jax.jit(jax.vmap(_sample_posture, (0,None,None,None)))

In [59]:
N = 100_000
key, subkey = jax.random.split(key)
keys = jax.random.split(subkey, N)
mask = robot.reach_mask(key, body, footholds)

%timeit -r 1 -n 1 jax.block_until_ready(sample_posture(keys, body, mask, ids))

oks, postures, supports = sample_posture(keys, body, mask, ids)

print(f"{oks.sum():,d}")

264 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
34,581


In [60]:
N = 100_000
key, subkey = jax.random.split(key)
keys = jax.random.split(subkey, N)




# TODO: we should make a sample posture function conditioned 
# on ankle limits and so on, so we can sample the legs independently
oks, postures, supports = sample_posture(keys, body, mask, ids)
order = jnp.argsort(oks.astype(jnp.float32))[::-1]

# =============================================
for t,i in enumerate(order[:10]):
    posture = postures[i]
    rrv.set_time(t, timeline="t")
    rrv.log_posture("posture", robot, posture,
                    foot_radius=0.025, disk_radius=0.025, 
                    disk_thickness=0.03, link_width=0.025, 
                    joint_values=posture.thetas, 
                    joint_cmap="bwr", 
                    joint_min=-np.pi, joint_max=np.pi,
                    body_color=(200, 200, 200), 
                    link_color=(200, 200, 200), 
                    )

In [61]:
N = 1_000
key, subkey = jax.random.split(key)
keys = jax.random.split(subkey, N)


oks, postures, supports = sample_posture(keys, body, mask, ids)

In [62]:
postures.shape

(1000,)

In [63]:
compute_forces = jax.jit(lambda qpos, support_masks: forces.stance_forces_batch(mjx_model, foot_ids, qpos, support_masks))
compute_sigs = jax.jit(lambda qpos, support_masks: forces.stance_sigma_min_batch(mjx_model, foot_ids, 
            qpos, support_masks, length=robot.leg.lengths.sum()))

In [64]:
support_masks = supports.mask(robot.num_legs)
qpos = jax.vmap(forces.to_qpos)(postures)  # assuming `postures` is  defined
frc, tau = compute_forces(qpos, support_masks)
sigma_min = compute_sigs(qpos, support_masks)

%timeit -r 1 -n 1 jax.block_until_ready(compute_forces(qpos, support_masks));
%timeit -r 1 -n 1 jax.block_until_ready(compute_sigs(qpos, support_masks));

8.21 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
1.71 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [65]:
def free_id(ids):
    return jnp.argmax(jnp.ones(4).at[ids].set(0))


def _sample_leg(key, body, mask, i):
    probs = mask.astype(jnp.float32)/mask.sum(axis=0, keepdims=True)
    shoulder = robot.shoulders(body)[i]

    f = jax.random.choice(key, footholds.shape[0], p=probs[:,i])

    return robot.leg.sample_planted(key, footholds[f].transform(shoulder.inverse()))


sample_leg = jax.jit(jax.vmap(_sample_leg, (0,None,None,None)))

In [66]:



posture = postures[0]
i = free_id(ids)

ok, theta = _sample_leg(key, posture.body, mask, i)

posture.thetas = posture.thetas.at[i].set(theta)



rrv.set_time(0)
rrv.log_posture("posture", robot, posture,
                foot_radius=0.025, disk_radius=0.025, 
                disk_thickness=0.03, link_width=0.025, 
                joint_values=posture.thetas, 
                joint_cmap="bwr", 
                joint_min=-np.pi, joint_max=np.pi,
                body_color=(200, 200, 200), 
                link_color=(200, 200, 200), 
                )

In [99]:
N = 10_000
key, subkey = jax.random.split(key)
keys = jax.random.split(subkey, N)

%timeit -r 1 -n 1 jax.block_until_ready(sample_leg(keys, body, mask, i))


oks, thetas = sample_leg(keys, body, mask, i)
print(f"{oks.sum():,d}", thetas.shape)

14.1 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)
7,363 (10000, 4)


2026-09-23T11:19:46.330645Z ERROR re_grpc_client::write: Write messages call failed: transport error


In [150]:
probs = mask.astype(jnp.float32)/mask.sum(axis=0, keepdims=True)
probs[:,i].sum()

Array(0.99999994, dtype=float32)

In [15]:
ankle_limit = jnp.deg2rad(45.0)
ankles_in_limit = compute_contact_angles(postures, supports) < ankle_limit
ankles_in_limit = ankles_in_limit.all(axis=-1)
has_collision = check_leg_collisions(postures, robot)
valid = ankles_in_limit & ~has_collision & oks

print(f"""
    OK postures: {oks.sum()} / {N}
    Ankles in limit: {ankles_in_limit.sum()} / {N}
    No self collisions: {(~has_collision).sum()} / {N}
    Valid postures: {valid.sum()} / {N}
""")

postures = postures[valid][:2000]
supports = supports[valid][:2000]
valid = valid[valid][:2000]


    OK postures: 34574 / 100000
    Ankles in limit: 27 / 100000
    No self collisions: 85945 / 100000
    Valid postures: 25 / 100000



In [19]:
support_masks = supports.mask(robot.num_legs)
qpos = jax.vmap(forces.to_qpos)(postures)  # assuming `postures` is defined
frc, tau = forces.stance_forces_batch(mjx_model, foot_ids, qpos, support_masks)
sigma_min = forces.stance_sigma_min_batch(mjx_model, foot_ids, 
            qpos, support_masks, length=robot.leg.lengths.sum())

In [15]:
from controlkit.utils import complement

wf, wt, ws = 1., 1., 1.
# wf, wt, ws = 0., 1., 0.
# wf, wt, ws = 1., 0., 0.
# wf, wt, ws = 0., 0., 1.

scores = (
    wf * jnp.exp(-jnp.linalg.norm(frc, axis=(-2,-1))) +
    wt * jnp.exp(-jnp.linalg.norm(tau, axis=(-2,-1))) + 
    ws * sigma_min 
)
scores = jnp.where(valid, scores, -jnp.inf)
order = jnp.argsort(scores)[::-1]


for t,i in enumerate(order[:10]):
    posture = postures[i]
    support = supports[i]

    posture.thetas = posture.thetas.at[complement(support.ids)].set(
        jnp.deg2rad(jnp.array([0.0, 0.0, -70.0, 160.0])))
    rrv.set_time(t, timeline="t")
    rrv.log_mesh("mesh", vertices, faces, color=(200, 200, 200))
    rrv.log_contacts("footholds", 
                 support.sites.position - 0.025*support.sites.normal, 
                 support.sites.normal, size=0.1,
                 thickness=0.01,
                 color=[(200,200,200)], 
                 normal_color=[(0, 0, 255)], show_normals=False)
    rrv.log_posture("posture", robot, posture, 
                    frc=frc[i], tau=tau[i], force_scale=0.01,
                    foot_radius=0.025, disk_radius=0.025, 
                    disk_thickness=0.03, link_width=0.025, 
                    joint_values=posture.thetas, 
                    joint_cmap="bwr", 
                    joint_min=-np.pi, joint_max=np.pi,
                    body_color=(200, 200, 200), 
                    link_color=(200, 200, 200), 
                    )
    

In [16]:
jnp.savez("_stable_postures.npz", postures=postures[order], supports=supports[order], robot=robot)

In [ ]:
from controlkit import mujoco_viz

i = order[0]
# mujoco_viz.show(robot, postures[i])  
img = mujoco_viz.render(robot, postures[i])

2026-08-24T12:17:45.719002Z ERROR re_grpc_client::write: Write messages call failed: transport error
